# Project — TGV punctuality by route — *Régularité mensuelle TGV par liaisons*
**MSA-DATI07-03 · Advanced EDA & Data Collection** · difficulty ⭐ easy

This is your **project notebook**. It already follows the six-section structure from Session 6 and contains working collection code for this API. Your job: fill every `# TODO` and every *TODO* markdown cell, on **real data you collected yourselves**.

| # | Section | Answers the question… |
|---|---|---|
| 1 | Context & provenance | Where does this data come from, and what question is it for? |
| 2 | Setup & parameters | Can someone else run this and get the same thing? |
| 3 | Collection, ingestion & schema | Did we collect it properly, and is it what we think it is? |
| 4 | Data quality | What is missing, duplicated or weird — and what did we decide? |
| 5 | Exploration | What does the data say? Every figure captioned. |
| 6 | Findings & next steps | What can a modeller trust, and what should they test? |

**Deliverables:** this notebook (Restart & Run All passes, same fingerprint twice) · `requirements.txt` · the 2-page data-quality report (`DATA_QUALITY_REPORT_TEMPLATE.md`) · regular Git commits.

**Possible questions** (pick one or write your own — one per group):
- Which routes are the least punctual, and is it getting better or worse over time?
- What share of delays comes from external causes vs infrastructure vs rolling stock?
- Do long routes have more delays, or just longer ones?


---
## 1 · Context & provenance

*TODO — complete this card before writing any analysis.*

| Field | Your answer |
|---|---|
| **Business question** | On French TGV routes (Jan 2021 – Sep 2026), are long routes late **more often** (share of trains arriving late) or late **by more minutes** when they are late (mean delay of late trains)? Answering this tells an operator or the regulator (AQST) whether punctuality efforts on long routes should aim at **reducing the number of delays** or **shortening them**. |
| **Source** | SNCF open-data portal (ressources.data.sncf.com), dataset `regularite-mensuelle-tgv-aqst` |
| **Dataset page / API docs** | https://ressources.data.sncf.com/explore/dataset/regularite-mensuelle-tgv-aqst/ |
| **Collected by / when** | Sarah BENADDAD Marion GOMES DE SOUSA Farida SAID · 2026-11-04 |
| **Collection method** | REST API (Opendatasoft Explore v2.1), paginated 100 rows/page, cached with `requests-cache` (see Section 3) |
| **Filters applied** | `WHERE = "date >= date'2021-01-01'"`, `ORDER_BY = "date"` → 8,395 rows (= `total_count` reported by the API, nothing truncated), months 2021-01 to 2026-09. No column filter at collection. |
| **Licence / constraints** | Open Database License (ODbL) — attribution required, derived databases must stay under ODbL |
| **Unit of observation** | One row = one route (`gare_depart` → `gare_arrivee`) for one month (`date`). Checked in Section 3: no duplicate on (`date`, `gare_depart`, `gare_arrivee`). |
| **Known limitations** | (1) **"Late" depends on trip length** (AQST composite definition): late = ≥ 5 min if trip < 1h30, ≥ 10 min if 1h30–3h, ≥ 15 min if > 3h. Long routes are therefore *less likely* to be counted late by construction — this directly biases our "more often late" comparison. (2) **Monthly aggregates only**: no train-level data, no median or distribution; `retard_moyen_*` are already means, so they must be weighted by number of trains when aggregated. (3) **A train can count in several routes**: rows cannot be summed to get a global or per-axis figure (SNCF warning). (4) **Delays rounded down to the minute**. (5) **COVID in early 2021** (curfew, April lockdown): atypical volumes and delays in H1 2021. (6) **Route length** only as `duree_moyenne` (mean travel time), not distance. (7) **Snapshot**: data as published on 2026-10-02, SNCF can revise past months; pre-2021 data excluded on purpose. |

---
## 2 · Setup & parameters
On Colab, run the install cell once per session.

In [9]:
# Colab / fresh environment only
import importlib.util
missing = [p for p in ("pandera", "requests_cache") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q "pandera>=0.24" requests-cache

In [10]:
# --- Parameters: the ONLY cell you should need to edit to rerun on new data ---
SEED = 42
DATA_DIR = "data"
REPORT_DIR = "reports"
CACHE_HOURS = 24 * 7                              # re-use API responses for a week
IQR_K = 1.5
MAX_MISSING_SHARE = 0.30

BASE_URL = "https://ressources.data.sncf.com"
DATASET_ID = "regularite-mensuelle-tgv-aqst"
WHERE = "date >= date'2021-01-01'"
ORDER_BY = "date"
MAX_ROWS = 10_000
PAUSE_S = 0.5

In [11]:
import hashlib, json, random, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandera
import pandera.pandas as pa
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

random.seed(SEED); np.random.seed(SEED)
DATA, REPORTS = Path(DATA_DIR), Path(REPORT_DIR)
for p in (DATA / "raw", DATA / "processed", REPORTS):
    p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

for lib in (pd, np, sns, pandera, requests):
    print(f"{lib.__name__:<10}{lib.__version__}")

pandas    2.1.4
numpy     1.26.4
seaborn   0.12.2
pandera   0.35.0
requests  2.31.0


In [12]:
# Write requirements.txt from the packages this notebook imports
from importlib.metadata import version
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "pandera", "pyarrow", "requests", "requests-cache"]
Path("requirements.txt").write_text("".join(f"{p}=={version(p)}\n" for p in pkgs))
print(Path("requirements.txt").read_text())

pandas==2.1.4
numpy==1.26.4
matplotlib==3.8.0
seaborn==0.12.2
pandera==0.35.0
pyarrow==14.0.2
requests==2.31.0
requests-cache==1.3.3



---
## 3 · Collection, ingestion & schema

### 3.1 A polite, cached HTTP session
- **Cache**: the same request is never sent twice (fast reruns, kind to the API).
- **Retry with backoff** on 429 / 5xx, respecting `Retry-After`.
- **Throttle**: pause between requests that actually hit the network.

In [13]:
session = requests_cache.CachedSession("cache/api", expire_after=CACHE_HOURS * 3600)
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True)))
session.headers["User-Agent"] = "MSA-DATI07-student-project/1.0"

def get(url, params=None):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    if not getattr(r, "from_cache", False):
        time.sleep(PAUSE_S)
    return r

### 3.2 Collect

In [14]:
def collect_ods(base_url, dataset_id, where=None, order_by=None, max_rows=MAX_ROWS, use_export=False):
    """Opendatasoft Explore API v2.1. Offset pagination, 100 rows per page.
    The API refuses offset + limit > 10,000: above that, filter with `where` or use the export endpoint."""
    url = f"{base_url}/api/explore/v2.1/catalog/datasets/{dataset_id}"
    params = {k: v for k, v in {"where": where, "order_by": order_by}.items() if v}
    if use_export:                                   # one call, no row cap, returns a JSON list
        r = get(f"{url}/exports/json", params=params)
        return r.json()[:max_rows]
    rows, offset, limit = [], 0, 100
    while offset < min(max_rows, 10_000):
        r = get(f"{url}/records", params={**params, "limit": limit, "offset": offset})
        page = r.json()
        if offset == 0:
            print(f"API reports {page['total_count']:,} matching rows")
        rows += page["results"]
        if len(page["results"]) < limit:             # last page
            break
        offset += limit
    return rows[:max_rows]

In [15]:
collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
raw_rows = collect_ods(BASE_URL, DATASET_ID, where=WHERE, order_by=ORDER_BY)
print(f"collected {len(raw_rows):,} rows at {collected_at}")

API reports 8,395 matching rows
collected 8,395 rows at 2026-11-04T11:17:15+00:00


### 3.3 Flatten, save raw Parquet + README

In [16]:
df_raw = pd.json_normalize(raw_rows, sep='_')

In [ ]:
# --- ajout groupe : check de la période collectée ---
# avec le WHERE on doit avoir de 2021-01 jusqu'au dernier mois publié
print(df_raw["date"].min(), "→", df_raw["date"].max())

('2021-01', '2026-09')

In [18]:
raw_path = DATA / "raw" / "raw.parquet"
df_raw.to_parquet(raw_path, index=False)
(DATA / "raw" / "README.md").write_text(f"""# Raw data
- Source: see Section 1 of the notebook
- Collected at (UTC): {collected_at}
- Rows x columns: {df_raw.shape[0]} x {df_raw.shape[1]}
- Columns: {", ".join(df_raw.columns)}
""")
print(df_raw.shape)
df_raw.head()

(8395, 26)


,date,service,gare_depart,gare_arrivee,duree_moyenne,nb_train_prevu,nb_annulation,commentaire_annulation,nb_train_depart_retard,retard_moyen_depart,...,nb_train_retard_sup_15,retard_moyen_trains_retard_sup15,nb_train_retard_sup_30,nb_train_retard_sup_60,prct_cause_externe,prct_cause_infra,prct_cause_gestion_trafic,prct_cause_materiel_roulant,prct_cause_gestion_gare,prct_cause_prise_en_charge_voyageurs
0,2021-01,International,BARCELONA,PARIS LYON,401,30,0,None,0,0.000000,...,2,18.333333,0,0,50.000000,0.000000,0.000000,0.000000,0.000000,50.0
1,2021-01,International,LAUSANNE,PARIS LYON,226,98,1,None,85,3.169412,...,6,33.141667,3,0,33.333333,16.666667,0.000000,16.666667,33.333333,0.0
2,2021-01,International,PARIS LYON,BARCELONA,399,31,0,None,0,0.000000,...,0,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,2021-01,International,PARIS LYON,ITALIE,425,62,1,None,1,1.816667,...,4,57.125000,3,1,0.000000,25.000000,25.000000,25.000000,0.000000,25.0
4,2021-01,International,ZURICH,PARIS LYON,244,102,0,None,45,1.745185,...,13,60.610256,9,4,30.769231,23.076923,23.076923,15.384615,7.692308,0.0


In [19]:
# First look: types, missing share, number of distinct values per column
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "share_missing": df_raw.isna().mean().round(3),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("share_missing", ascending=False)
overview

,dtype,share_missing,n_unique
commentaire_annulation,object,1.0,0
commentaire_retards_depart,object,1.0,0
commentaires_retard_arrivee,object,1.0,0
date,object,0.0,69
retard_moyen_tous_trains_arrivee,float64,0.0,8291
prct_cause_gestion_gare,float64,0.0,2140
prct_cause_materiel_roulant,float64,0.0,2363
prct_cause_gestion_trafic,float64,0.0,2254
prct_cause_infra,float64,0.0,2240
prct_cause_externe,float64,0.0,2478


In [27]:
# --- ajout groupe : check de l'unité d'observation ---
# au début on pensait 1 ligne = 1 liaison x 1 mois, mais y'a des doublons sans le service
key_sans_service = ["date", "gare_depart", "gare_arrivee"]
key = ["date", "service", "gare_depart", "gare_arrivee"]
print("doublons sans service :", df_raw.duplicated(key_sans_service).sum())   # 42
print("doublons avec service :", df_raw.duplicated(key).sum())                # 0 -> c'est ça la vraie clé

# les 42 c'est des paires National / International, que sur juillet-sept 2025 (14 liaisons/mois)
dups = df_raw[df_raw.duplicated(key_sans_service, keep=False)]
print(dups.groupby(["date", "service"]).size())

# et ces 14 liaisons sont en National tout le reste du temps -> anomalie de publication
# on décide quoi en faire en 4.1
liaisons = dups[["gare_depart", "gare_arrivee"]].drop_duplicates()
print(df_raw.merge(liaisons, on=["gare_depart", "gare_arrivee"])["service"].value_counts())

doublons sans service : 42
doublons avec service : 0
date     service      
2025-07  International    14
         National         14
2025-08  International    14
         National         14
2025-09  International    14
         National         14
dtype: int64
service
National         966
International     42
Name: count, dtype: int64


### 3.4 Schema-as-contract

*TODO (Exercise 3 of Session 6)*: write a `pa.DataFrameSchema` for the columns you keep. Requirements: every kept column with a dtype, `strict=True`, **at least 4 domain checks**, `nullable=True` only where justified in a comment. Validate the raw data with `lazy=True` and keep the failure summary for your report.

Columns to look at first (confirm the exact names in `df_raw.columns`): `date`, `service`, `gare_depart`, `gare_arrivee`, `duree_moyenne`, `nb_train_prevu`, `nb_annulation`, `nb_train_retard_arrivee`, `retard_moyen_arrivee` and the `prct_cause_*` columns.

In [29]:
# TODO: select the columns you keep, then write the contract
# KEEP = list(df_raw.columns)          # TODO: replace by your selection (and log it as a decision)

# colonnes gardées pour notre question (longs trajets = + de retards ou retards + longs ?)
KEEP = [
    # clé
    "date", "service", "gare_depart", "gare_arrivee",
    # longueur du trajet (en min, pas de distance dispo)
    "duree_moyenne",
    # fréquence -> % en retard = retard / (prévus - annulés)
    "nb_train_prevu", "nb_annulation", "nb_train_retard_arrivee",
    # ampleur -> retard moyen des trains EN RETARD seulement
    # (pas tous_trains, ça mélange fréquence et ampleur)
    "retard_moyen_arrivee",
    # gros retards + pour tester les seuils AQST 5/10/15 min en partie 4
    "nb_train_retard_sup_15", "nb_train_retard_sup_30", "nb_train_retard_sup_60",
]
# virés : départ (l'AQST regarde l'arrivée), causes (= question 2 pas la nôtre),
# commentaires (texte libre), retard_moyen_trains_retard_sup15 (redondant)

df_sel = df_raw[KEEP].copy()

#schema = None                        # TODO: replace None by pa.DataFrameSchema({...}, strict=True)

# pas de nullable=True : 0% de manquants sur les colonnes gardées (cf overview)
schema = pa.DataFrameSchema(
    {
        "date": pa.Column(str, pa.Check.str_matches(r"^\d{4}-\d{2}$"), nullable=False),   # format AAAA-MM
        "service": pa.Column(str, pa.Check.isin(["National", "International"]), nullable=False),
        "gare_depart": pa.Column(str, pa.Check.str_length(min_value=1), nullable=False),
        "gare_arrivee": pa.Column(str, pa.Check.str_length(min_value=1), nullable=False),
        "duree_moyenne": pa.Column(int, pa.Check.in_range(1, 600), nullable=False),        # trajet TGV < 10h
        "nb_train_prevu": pa.Column(int, pa.Check.ge(0), nullable=False),                 # nb de trains >= 0
        "nb_annulation": pa.Column(int, pa.Check.ge(0), nullable=False),
        "nb_train_retard_arrivee": pa.Column(int, pa.Check.ge(0), nullable=False),
        "retard_moyen_arrivee": pa.Column(float, pa.Check.ge(0), nullable=False),         # retard moyen >= 0
        "nb_train_retard_sup_15": pa.Column(int, pa.Check.ge(0), nullable=False),
        "nb_train_retard_sup_30": pa.Column(int, pa.Check.ge(0), nullable=False),
        "nb_train_retard_sup_60": pa.Column(int, pa.Check.ge(0), nullable=False),
    },
    checks=[
        # pas plus d'annulations que de trains prévus
        pa.Check(lambda d: d["nb_annulation"] <= d["nb_train_prevu"], name="annul_le_prevu"),
        # en retard <= trains qui ont vraiment roulé
        pa.Check(lambda d: d["nb_train_retard_arrivee"] <= d["nb_train_prevu"] - d["nb_annulation"],
                 name="retard_le_circules"),
        # paliers emboîtés : +60 min <= +30 <= +15
        pa.Check(lambda d: (d["nb_train_retard_sup_60"] <= d["nb_train_retard_sup_30"])
                         & (d["nb_train_retard_sup_30"] <= d["nb_train_retard_sup_15"]),
                 name="paliers_emboites"),
        # seuil AQST max = 15 min, donc un train à +15 est forcément compté en retard
        pa.Check(lambda d: d["nb_train_retard_sup_15"] <= d["nb_train_retard_arrivee"],
                 name="sup15_le_retard"),
    ],
    unique=["date", "service", "gare_depart", "gare_arrivee"],   # clé validée en 3.3
    strict=True,
)

if schema is not None:
    try:
        schema.validate(df_sel, lazy=True)
        print("✅ raw data passes the contract")
    except pa.errors.SchemaErrors as err:
        fails = err.failure_cases
        print(f"❌ {len(fails)} violations")
        display(fails.groupby(["column", "check"]).size().rename("n").reset_index())

❌ 4303 violations


,column,check,n
0,date,paliers_emboites,355
1,duree_moyenne,paliers_emboites,355
2,gare_arrivee,paliers_emboites,355
3,gare_depart,paliers_emboites,355
4,nb_annulation,paliers_emboites,355
5,nb_train_prevu,paliers_emboites,355
6,nb_train_retard_arrivee,paliers_emboites,355
7,nb_train_retard_sup_15,paliers_emboites,355
8,nb_train_retard_sup_30,greater_than_or_equal_to(0),43
9,nb_train_retard_sup_30,paliers_emboites,355


In [30]:
# --- ajout groupe : résumé lisible des violations (en lignes, pas en cellules) ---
# les checks multi-colonnes comptent 1 violation par colonne -> on compte les lignes uniques
fails.groupby("check")["index"].nunique().rename("n_lignes")

check
greater_than_or_equal_to(0)     43
paliers_emboites               355
Name: n_lignes, dtype: int64

---
## 4 · Data quality — measure → decide → log

Every decision gets one `log(...)` call. The table at the end of this section goes into your report.

**What to look for in this dataset:**
- Missing months for some routes (COVID 2020, strikes): is that MNAR? Can you compare years fairly?
- Station names that change spelling over time: duplicates in disguise. Build a mapping and log it.
- Do the `prct_cause_*` shares add up to 100%? Check row by row.
- Means of means: `retard_moyen_arrivee` is already an average — weight by number of trains when aggregating.

In [ ]:
decisions = []
def log(column, issue, n_rows, decision, why):
    decisions.append(dict(column=column, issue=issue, n_rows=n_rows, decision=decision, why=why))

df = df_sel.copy()

### 4.1 Duplicates

In [ ]:
n_exact = int(df.astype(str).duplicated().sum())    # astype(str): some columns may hold dicts
print("exact duplicate rows:", n_exact)
# TODO: decide, apply, and log. Then look for near-duplicates (same entity, different id or spelling).

### 4.2 Impossible values

In [ ]:
# TODO: for each numeric column, what is physically or logically impossible? Count, decide, log.
df.describe().T

### 4.3 Missingness — mechanism before method

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0].rename("share_missing").to_frame()
miss["flag"] = np.where(miss.share_missing > MAX_MISSING_SHARE, "⚠️ high", "")
miss

In [ ]:
# TODO: for the 2–3 most important columns with missing values, test whether missingness
# depends on another observed column (MAR) — e.g.
# df.assign(m=df["<col>"].isna()).groupby("<other col>")["m"].mean()
# Then name the mechanism (MCAR / MAR / MNAR), decide, log.

### 4.4 Outliers — triage, don't delete

In [ ]:
# TODO: pick your key numeric column. Use an IQR fence (on a log scale if it is skewed),
# look at the flagged rows, separate errors from real extremes, decide per group, log.

In [ ]:
decision_table = pd.DataFrame(decisions, columns=["column", "issue", "n_rows", "decision", "why"])
decision_table.to_csv(REPORTS / "decision_table.csv", index=False)
decision_table

---
## 5 · Exploration — every figure gets a caption and a *so what*

In [ ]:
def show(fig, title, caption, so_what, name):
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.text(0.01, -0.04, f"{caption}\n→ {so_what}", ha="left", va="top", fontsize=9, wrap=True)
    fig.savefig(REPORTS / f"{name}.png", bbox_inches="tight", dpi=120)
    plt.show()

clean = df.copy()      # TODO: this should be your cleaned data after Section 4

### 5.1 Univariate

In [ ]:
# TODO: one figure showing the distribution of your key variable (ECDF before `.describe()`)
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_1")

### 5.2 Bivariate

In [ ]:
# TODO: one figure showing your key variable against 1–2 explanatory variables
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_2")

### 5.3 Multivariate

In [ ]:
# TODO: one figure showing the correlation structure, or a breakdown by two categories
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_3")

---
## 6 · Findings, what cannot be trusted, next steps

*TODO*

**Three findings** — each with the figure it comes from and what it means for modelling.
1.
2.
3.

**What cannot be trusted** — column, subset, and why.
-
-

**Five testable hypotheses** — variable, direction, population, test.
- H1:
- H2:
- H3:
- H4:
- H5:

### Final contract & fingerprint
*TODO*: write `clean_schema` for the data a modeller receives (it may differ from the raw schema). Then **Restart & Run All twice**: the fingerprint must be identical.

In [ ]:
clean_schema = None                  # TODO: pa.DataFrameSchema({...}, strict=True)
if clean_schema is not None:
    clean_schema.validate(clean, lazy=True)
    print("✅ clean data passes the contract")

clean.to_parquet(DATA / "processed" / "clean.parquet", index=False)
fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(clean.astype(str), index=False).values.tobytes()).hexdigest()[:12]
print("rows:", len(clean), "· fingerprint:", fingerprint)

> **Reproducibility note.** The fingerprint is stable between runs *thanks to the cache*: the API data changes over time, so a run without the cache (or after `CACHE_HOURS`) can give a different fingerprint. That is expected — record the collection date in Section 1 and in the README, and commit the raw Parquet if it is small enough.